# 06 Estimate the survival model

This notebook estimates a Cox proportional-hazards model for the duration of firm records. The event is a non-missing `Mitglied_Löschdatum`, stored upstream as `exit_date`; no dormant-status field is available in the current analytical file.

The data use counting-process intervals: one row per firm and quarter at risk. Firm age is the model time, and calendar year is a time-varying covariate. Firms already active in 2015 enter at their observed age (left truncation).

Run this notebook only after routing notebooks 04 and 05 have finished for all years. The preflight stops if any yearly firm-accessibility product is absent or incompatible.

## Configuration

In [8]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "TOOLS").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or one of its subdirectories.")


PROJECT_DIR = discover_project_dir()
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
FEATURE_ROOT = ANAL_DATA / "routing" / "features"
FIRMS_PATH = ANAL_DATA / "firms_assigned_100m.geoparquet"
MODEL_DIR = ANAL_DATA / "models"

START_YEAR, END_YEAR = 2015, 2025
YEARS = list(range(START_YEAR, END_YEAR + 1))
CAR_CONTOURS = [5, 10, 15, 30]

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

## 1 Preflight

In [9]:
firm_required = {"firm_id", "founding_date", "exit_date", "exit_observed", "grid_id_100m", "Fachgruppe_ID"}
access_required = {
    "firm_id", "grid_id_100m", "Fachgruppe_ID", "year", "quarter", "period",
    "included_in_lagged_stock", "own_cell_pop", "own_cell_firms", "own_cell_same_fachgruppe_firms",
}
access_required |= {f"pop_access_{m}min" for m in CAR_CONTOURS}
access_required |= {f"existing_firms_access_{m}min" for m in CAR_CONTOURS}
access_required |= {f"same_fachgruppe_firms_access_{m}min" for m in CAR_CONTOURS}

required_files = {FIRMS_PATH: firm_required}
for year in YEARS:
    required_files[FEATURE_ROOT / str(year) / "firm_accessibility_quarter_100m.parquet"] = access_required

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Routing is not complete. Missing files: {missing_files}")

for path, required_columns in required_files.items():
    columns = set(pq.ParquetFile(path).schema_arrow.names)
    missing_columns = sorted(required_columns - columns)
    if missing_columns:
        raise ValueError(f"{path.name} is missing columns: {missing_columns}")

print(f"Preflight passed for {len(YEARS)} routing years.")

Preflight passed for 11 routing years.


## 2 Load firm-quarter records

In [10]:
firm_glob = (FEATURE_ROOT / "*" / "firm_accessibility_quarter_100m.parquet").as_posix()
access_columns = ", ".join(
    column
    for m in CAR_CONTOURS
    for column in [f"pop_access_{m}min", f"existing_firms_access_{m}min", f"same_fachgruppe_firms_access_{m}min"]
)

spells = con.execute(
    f"""
    SELECT firm_id, grid_id_100m AS grid_id, Fachgruppe_ID, year, quarter, period,
           included_in_lagged_stock, own_cell_pop, own_cell_firms, own_cell_same_fachgruppe_firms,
           {access_columns}
    FROM read_parquet('{firm_glob}')
    WHERE year BETWEEN {START_YEAR} AND {END_YEAR}
    ORDER BY firm_id, year, quarter
    """
).df()

assert not spells.duplicated(["firm_id", "year", "quarter"]).any()
print(f"{len(spells):,} firm-quarter rows, {spells['firm_id'].nunique():,} firms")

400,618 firm-quarter rows, 9,783 firms


### Remove incomplete histories

A skipped routing origin can leave accessibility values missing. The whole firm history is removed, rather than deleting isolated intervals and creating gaps in its risk history.

In [11]:
routing_columns = [
    *[f"pop_access_{m}min" for m in CAR_CONTOURS],
    *[f"existing_firms_access_{m}min" for m in CAR_CONTOURS],
    *[f"same_fachgruppe_firms_access_{m}min" for m in CAR_CONTOURS],
]
incomplete_firms = spells.loc[spells[routing_columns].isna().any(axis=1), "firm_id"].unique()
missing_group_firms = spells.loc[spells["Fachgruppe_ID"].isna(), "firm_id"].unique()
excluded_firms = np.union1d(incomplete_firms, missing_group_firms)
print(f"Excluded firms with incomplete routing or Fachgruppe data: {len(excluded_firms):,}")
spells = spells.loc[~spells["firm_id"].isin(excluded_firms)].copy()

Excluded firms with incomplete routing or Fachgruppe data: 0


## 3 Build event intervals

In [12]:
firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", "founding_date", "exit_date", "exit_observed"])
firms["founding_date"] = pd.to_datetime(firms["founding_date"], errors="coerce")
firms["exit_date"] = pd.to_datetime(firms["exit_date"], errors="coerce")
spells = spells.merge(firms, on="firm_id", how="left", validate="many_to_one")
assert spells["founding_date"].notna().all(), "A routed firm has no founding date."

period_index = pd.PeriodIndex(spells["period"], freq="Q").asi8
founding_index = spells["founding_date"].dt.to_period("Q").array.asi8
spells["start"] = period_index - founding_index
spells["stop"] = spells["start"] + 1
exit_period = spells["exit_date"].dt.to_period("Q").astype("string")
spells["event"] = (spells["exit_observed"].fillna(False) & spells["period"].eq(exit_period)).astype("int8")
spells = spells.sort_values(["firm_id", "year", "quarter"])

assert (spells["start"] >= 0).all()
assert (spells["stop"] > spells["start"]).all()
assert (spells.groupby("firm_id")["event"].sum() <= 1).all()
period_number = spells["year"] * 4 + spells["quarter"]
gaps = period_number.groupby(spells["firm_id"]).diff().dropna().ne(1)
assert not gaps.any(), "Firm-quarter histories contain gaps."

print(f"Observed exits: {spells['event'].sum():,}")
print(f"Median age at study entry: {spells.groupby('firm_id')['start'].min().median():.0f} quarters")

Observed exits: 1,495
Median age at study entry: 89 quarters


## 4 Build covariates

The focal firm is removed from the lagged firm masses when it was already active at the start of the quarter. Own-cell masses are then separated from the wider 0–5 minute ring. Same-Fachgruppe and other-Fachgruppe masses are used instead of also including their total, which avoids an additive duplication.

In [13]:
focal = spells["included_in_lagged_stock"].astype("float64")
spells["own_firms"] = (spells["own_cell_firms"] - focal).clip(lower=0)
spells["own_same"] = (spells["own_cell_same_fachgruppe_firms"] - focal).clip(lower=0)
spells["own_other"] = (spells["own_firms"] - spells["own_same"]).clip(lower=0)

for m in CAR_CONTOURS:
    spells[f"firms_{m}"] = (spells[f"existing_firms_access_{m}min"] - focal).clip(lower=0)
    spells[f"same_{m}"] = (spells[f"same_fachgruppe_firms_access_{m}min"] - focal).clip(lower=0)
    spells[f"other_{m}"] = (spells[f"firms_{m}"] - spells[f"same_{m}"]).clip(lower=0)

def add_log_rings(frame: pd.DataFrame, source, prefix: str, contours: list[int], own_column: str) -> list[str]:
    cumulative = {m: np.clip(frame[source(m)].to_numpy(dtype="float64") - frame[own_column].to_numpy(dtype="float64"), 0, None) for m in contours}
    columns = []
    lower = 0
    for m in contours:
        previous = 0.0 if lower == 0 else cumulative[lower]
        column = f"log_{prefix}_ring_{lower}_{m}"
        frame[column] = np.log1p(np.clip(cumulative[m] - previous, 0, None))
        columns.append(column)
        lower = m
    return columns

population_rings = add_log_rings(spells, lambda m: f"pop_access_{m}min", "pop", CAR_CONTOURS, "own_cell_pop")
same_rings = add_log_rings(spells, lambda m: f"same_{m}", "same", CAR_CONTOURS, "own_same")
other_rings = add_log_rings(spells, lambda m: f"other_{m}", "other", CAR_CONTOURS, "own_other")

spells["log_own_pop"] = np.log1p(spells["own_cell_pop"])
spells["log_own_same"] = np.log1p(spells["own_same"])
spells["log_own_other"] = np.log1p(spells["own_other"])
spells["calendar_year"] = spells["year"] - START_YEAR

covariates = [
    "log_own_pop", *population_rings,
    "log_own_same", "log_own_other", *same_rings, *other_rings,
    "calendar_year",
]
print(covariates)

['log_own_pop', 'log_pop_ring_0_5', 'log_pop_ring_5_10', 'log_pop_ring_10_15', 'log_pop_ring_15_30', 'log_own_same', 'log_own_other', 'log_same_ring_0_5', 'log_same_ring_5_10', 'log_same_ring_10_15', 'log_same_ring_15_30', 'log_other_ring_0_5', 'log_other_ring_5_10', 'log_other_ring_10_15', 'log_other_ring_15_30', 'calendar_year']


## 5 Estimate the Cox model

`PHReg` accepts start/stop data through `entry` and `endog`. Standard errors are clustered by 100 m cell; this also accounts for repeated quarterly rows belonging to firms at the same fixed location. Efron's method handles the many quarterly ties.

Statsmodels marks score residuals as `NaN` for intervals outside every observed-event risk set. Those intervals contribute zero to the score. The notebook therefore replaces only those `NaN` values with zero before aggregating the score by cell.

In [14]:
from scipy import stats
from statsmodels.duration.hazard_regression import PHReg

model_frame = spells[["firm_id", "grid_id", "start", "stop", "event", *covariates]].copy()
assert np.isfinite(model_frame[covariates].to_numpy(dtype="float64")).all()
assert model_frame["event"].sum() > 0

cox_model = PHReg(
    endog=model_frame["stop"].to_numpy(dtype="float64"),
    exog=model_frame[covariates].astype("float64"),
    status=model_frame["event"].to_numpy(),
    entry=model_frame["start"].to_numpy(dtype="float64"),
    ties="efron",
)
cox_result = cox_model.fit()


def cluster_covariance(model, params, groups):
    """Clustered sandwich covariance with zero for rows outside every event risk set."""
    score_observations = model.score_residuals(params)
    if np.isinf(score_observations).any():
        raise ValueError("Infinite score residuals encountered.")
    never_at_risk = np.isnan(score_observations).any(axis=1)
    score_observations = np.nan_to_num(score_observations, nan=0.0)

    codes, cluster_labels = pd.factorize(groups, sort=False)
    if (codes < 0).any():
        raise ValueError("Missing grid_id values cannot be used for clustered covariance.")
    grouped_scores = np.zeros((len(cluster_labels), score_observations.shape[1]))
    np.add.at(grouped_scores, codes, score_observations)

    bread = np.linalg.inv(model.hessian(params))
    covariance = bread @ (grouped_scores.T @ grouped_scores) @ bread
    return covariance, int(never_at_risk.sum())


cluster_cov, n_bad = cluster_covariance(
    cox_model, cox_result.params, model_frame["grid_id"].to_numpy()
)
cluster_variances = np.diag(cluster_cov)
if (cluster_variances < -1e-12).any():
    raise ValueError("Clustered covariance has a negative diagonal element.")
cluster_bse = np.sqrt(np.clip(cluster_variances, 0.0, None))
cluster_z = np.asarray(cox_result.params) / cluster_bse
cluster_pvalues = 2 * stats.norm.sf(np.abs(cluster_z))
critical_value = stats.norm.ppf(0.975)
cluster_confidence = np.column_stack([
    np.asarray(cox_result.params) - critical_value * cluster_bse,
    np.asarray(cox_result.params) + critical_value * cluster_bse,
])

print("Fit summary (naive covariance; exported inference uses the clustered covariance below):")
print(cox_result.summary())
print("Naive standard errors:", np.round(np.asarray(cox_result.bse), 4))
print("Cell-clustered standard errors:", np.round(cluster_bse, 4))
zero_score_share = n_bad / len(model_frame)
print(f"Rows outside every event risk set: {n_bad:,} ({zero_score_share:.2%})")
if zero_score_share >= 0.10:
    print("Note: at least 10% of intervals make no score contribution; report this with the event-age distribution.")

Fit summary (naive covariance; exported inference uses the clustered covariance below):
                              Results: PHReg
Model:                        PH Reg          Sample size:          400603
Dependent variable:           y               Num. events:          -41   
Ties:                         Efron                                       
--------------------------------------------------------------------------
                      log HR log HR SE   HR      t    P>|t|  [0.025 0.975]
--------------------------------------------------------------------------
log_own_pop           0.0447    0.0231 1.0457  1.9371 0.0527 0.9995 1.0941
log_pop_ring_0_5      0.0315    0.0406 1.0320  0.7753 0.4381 0.9530 1.1176
log_pop_ring_5_10    -0.1036    0.0580 0.9016 -1.7865 0.0740 0.8047 1.0101
log_pop_ring_10_15    0.0089    0.0655 1.0089  0.1352 0.8924 0.8874 1.1470
log_pop_ring_15_30    0.1102    0.0969 1.1165  1.1380 0.2551 0.9235 1.3500
log_own_same         -0.7582    0.7072 0.4

d:\CO2_Masterarbeit\CO2_Masterarbeit\.venv\Lib\site-packages\statsmodels\duration\hazard_regression.py:1613: RuntimeWarning: overflow encountered in scalar add
  info["Num. events:"] = str(int(sum(self.model.status)))


## 6 Save results

In [15]:
results = pd.DataFrame(
    {
        "coefficient": np.asarray(cox_result.params),
        "std_error": cluster_bse,
        "p_value": cluster_pvalues,
        "ci_lower": cluster_confidence[:, 0],
        "ci_upper": cluster_confidence[:, 1],
    },
    index=covariates,
)
results["hazard_ratio"] = np.exp(results["coefficient"])
results["hr_ci_lower"] = np.exp(results["ci_lower"])
results["hr_ci_upper"] = np.exp(results["ci_upper"])

MODEL_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(MODEL_DIR / "survival_cox_results.csv", index_label="term")
pd.Series(
    {
        "n_intervals": len(model_frame),
        "n_firms": model_frame["firm_id"].nunique(),
        "n_events": int(model_frame["event"].sum()),
        "n_cells": model_frame["grid_id"].nunique(),
        "n_zero_score_rows": n_bad,
        "zero_score_row_share": zero_score_share,
    }
).to_csv(MODEL_DIR / "survival_cox_diagnostics.csv", header=["value"])
results.round(4)

,coefficient,std_error,p_value,ci_lower,ci_upper,hazard_ratio,hr_ci_lower,hr_ci_upper
log_own_pop,0.0447,0.1507,0.7667,-0.2507,0.3401,1.0457,0.7783,1.4051
log_pop_ring_0_5,0.0315,0.2479,0.8989,-0.4544,0.5174,1.0320,0.6348,1.6777
log_pop_ring_5_10,-0.1036,0.3187,0.7451,-0.7283,0.5211,0.9016,0.4827,1.6838
log_pop_ring_10_15,0.0089,0.3420,0.9794,-0.6615,0.6792,1.0089,0.5161,1.9722
log_pop_ring_15_30,0.1102,0.5386,0.8378,-0.9454,1.1659,1.1165,0.3885,3.2087
log_own_same,-0.7582,0.8443,0.3692,-2.4129,0.8965,0.4685,0.0896,2.4511
log_own_other,0.0280,0.6459,0.9655,-1.2380,1.2939,1.0284,0.2900,3.6471
log_same_ring_0_5,0.0058,0.4448,0.9897,-0.8661,0.8776,1.0058,0.4206,2.4052
log_same_ring_5_10,-0.0642,0.3529,0.8556,-0.7559,0.6274,0.9378,0.4696,1.8728
log_same_ring_10_15,-0.0354,0.3353,0.9159,-0.6926,0.6218,0.9652,0.5003,1.8623


## Interpretation notes

A hazard ratio below 1 indicates a lower exit rate and therefore longer survival. Before final interpretation, inspect convergence, high correlations, and time interactions for covariates that may violate proportional hazards. If the zero-score row share reaches double digits, report that the observed exits are thinly distributed over the age axis. The current source defines exit from `Mitglied_Löschdatum`; a different event definition requires rebuilding the upstream `exit_date`.